# 🔬 Pipeline Modeling & Baseline Eksperimen 3 Kelas (Multiclass)
### Dataset Irisan Bebas Kebocoran: HAM10000 ∩ ISIC 2017 ∩ ISIC 2019
**Diagnosis Target:** `NV` (Nevus / 0), `MEL` (Melanoma / 1), `BKL` (Benign Keratosis / 2)  
**Total Dataset:** 22.051 citra (Train: 17.656 | Val: 2.192 | Test: 2.203) — *Lesion-Aware Stratified Split*

---
Notebook ini menyediakan pipeline deep learning lengkap (PyTorch) berstandar riset ilmiah internasional:
1. **Pemeriksaan Komputasi & Hyperparameter Konfigurasi** (Reproducibility Seed 42)
2. **Pemuatan Partisi Bebas Kebocoran & Perhitungan Class Weights** (Penanganan Imbalance)
3. **Custom PyTorch Dataset & Transformasi Augmentasi Citra Dermatologi**
4. **Visualisasi Batch Inspeksi Augmentasi Citra**
5. **Inisialisasi Arsitektur Deep Learning Baseline (Transfer Learning: ResNet-50 & EfficientNet-B0)**
6. **Loss Function Berbobot (Weighted Cross Entropy / Focal Loss) & Optimizer AdamW**
7. **Modular Training & Validation Engine dengan Model Checkpointing (Best Val Macro F1)**
8. **Evaluasi Komprehensif pada Test Set (Unseen Data: Accuracy, Balanced Acc, Macro F1, Sensitivity, Specificity, ROC-AUC)**
9. **Visualisasi Diagnostik Medis (Normalized Confusion Matrix & ROC Curve OvR)**
10. **Penyimpanan Checkpoint Model & Ekspor Ringkasan Metrik**


## Langkah 1 — Setup Environment, Komputasi & Hyperparameter Reproducibility
Menyiapkan pustaka inti (`torch`, `torchvision`, `sklearn`, `matplotlib`), mendeteksi akselerator hardware (CUDA GPU vs CPU), dan menetapkan seed acak agar hasil eksperimen dapat direproduksi (*reproducible*).


In [ ]:
import os
import sys
import time
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, confusion_matrix,
    roc_auc_score, roc_curve, classification_report
)

# 1. Reproducibility Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Deteksi Perangkat (Device)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Menggunakan Perangkat Komputasi: {device}")
if device.type == "cuda":
    print(f"   Nama GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Tersedia: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("   ℹ️ Berjalan dalam mode CPU.")

# 3. Hyperparameter Konfigurasi Baseline
CONFIG = {
    "batch_size": 32,
    "image_size": 224,
    "num_classes": 3,
    "learning_rate": 1e-4,
    "weight_decay": 1e-2,
    "epochs": 10,
    "num_workers": 0, # Diset 0 untuk stabilitas Windows DataLoader
    "checkpoint_dir": "../models" if os.path.exists("../Dataset") else "models",
    "class_names": ["NV", "MEL", "BKL"],
    "class_fullnames": ["Melanocytic Nevus", "Melanoma", "Benign Keratosis"]
}

    "checkpoint_dir": "../models" if os.path.exists("../Dataset") else "models",
print("⚙️ Konfigurasi Hyperparameter siap:", CONFIG)


## Langkah 2 — Pemuatan Dataset Partisi & Perhitungan Class Weights
Membaca file partisi yang telah dibuat pada notebook pemetaan data (`dataset_irisan_3kelas_train.csv`, `val.csv`, `test.csv`).  
Menghitung frekuensi kelas dan menghitung **Inverse Class Weights**:
$$w_c = \frac{N_{total}}{K \times N_c}$$
untuk mencegah model bias terhadap kelas mayoritas (`NV` = 64,2%).


In [ ]:
# Deteksi Path Relatif (Adaptif terhadap working directory root maupun folder kode/)
BASE_DATA_DIR = '../../Dataset' if os.path.exists('../../Dataset') else ('../Dataset' if os.path.exists('../Dataset') else 'Dataset')
BASE_MODEL_DIR = '../../models' if os.path.exists('../../models') else ('../models' if os.path.exists('../models') else 'models')
os.makedirs(BASE_MODEL_DIR, exist_ok=True)

TRAIN_CSV = os.path.join(BASE_DATA_DIR, 'dataset_irisan_3kelas_train.csv')
VAL_CSV   = os.path.join(BASE_DATA_DIR, 'dataset_irisan_3kelas_val.csv')
TEST_CSV  = os.path.join(BASE_DATA_DIR, 'dataset_irisan_3kelas_test.csv')

df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(' Rekapitulasi Dataset Irisan 3 Kelas:')
print(f'   Training Set   : {len(df_train):,} citra ({len(df_train)/22051*100:.2f}%)')
print(f'   Validation Set : {len(df_val):,} citra ({len(df_val)/22051*100:.2f}%)')
print(f'   Test Set       : {len(df_test):,} citra ({len(df_test)/22051*100:.2f}%)')
print(f'   Total          : {len(df_train)+len(df_val)+len(df_test):,} citra')

# Distribusi Kelas pada Training Set
train_counts = df_train['target_multiclass'].value_counts().sort_index()
total_train = len(df_train)
num_classes = CONFIG['num_classes']

# Perhitungan Balanced Class Weights (Formula Sklearn / Standard Medis)
class_weights = total_train / (num_classes * train_counts.values)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print('\n Distribusi Kelas & Bobot Penyeimbang (Class Weights) Training:')
for idx, (cls_name, full_name) in enumerate(zip(CONFIG['class_names'], CONFIG['class_fullnames'])):
    count = train_counts[idx]
    pct = (count / total_train) * 100
    w = class_weights[idx]
    print(f'   Kelas {idx} [{cls_name} - {full_name}]: {count:,} citra ({pct:.1f}%) | Weight: {w:.4f}')


## Langkah 3 — Custom PyTorch Dataset & Pipeline Augmentasi Dermatologi
Membangun `SkinLesionDataset` dengan pipeline augmentasi citra:
* **Training Transform:** `RandomResizedCrop(224)`, `RandomHorizontalFlip()`, `RandomVerticalFlip()`, `RandomRotation(20)`, `ColorJitter(0.1, 0.1, 0.1)`, `Normalize(ImageNet)`.
* **Validation & Test Transform:** `Resize(256)`, `CenterCrop(224)`, `Normalize(ImageNet)`.


In [ ]:
class SkinLesionDataset(Dataset):
    """
    Custom Dataset PyTorch untuk Klasifikasi Citra Lesi Kulit ISIC & HAM10000.
    """
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.filepaths = self.df['filepath'].values
        self.labels = self.df['target_multiclass'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (CONFIG['image_size'], CONFIG['image_size']), color=(0, 0, 0))
        
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# Pipeline Augmentasi & Normalisasi Standar ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(CONFIG['image_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Buat Dataset Objek
train_dataset = SkinLesionDataset(df_train, transform=train_transforms)
val_dataset   = SkinLesionDataset(df_val, transform=eval_transforms)
test_dataset  = SkinLesionDataset(df_test, transform=eval_transforms)

# Inisialisasi DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=(device.type == 'cuda')
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=(device.type == 'cuda')
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=(device.type == 'cuda')
)

print(f"📦 DataLoaders Berhasil Diinisialisasi:")
print(f"   Train Batches : {len(train_loader)} (Batch Size: {CONFIG['batch_size']})")
print(f"   Val Batches   : {len(val_loader)}")
print(f"   Test Batches  : {len(test_loader)}")


## Langkah 4 — Visualisasi Batch Inspeksi Augmentasi Citra
Menampilkan satu batch citra hasil augmentasi untuk memverifikasi kualitas transformasi spatial dan un-normalisasi citra RGB.


In [ ]:
def unnormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Mengembalikan tensor normalisasi ke skala citra asli [0, 1]"""
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return torch.clamp(tensor, 0, 1)

# Ambil 1 batch sampel
images_batch, labels_batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Sampel Batch Citra Training (Dengan Augmentasi Spatial & Normalisasi)", fontsize=14, fontweight='bold')

for i in range(8):
    ax = axes[i // 4, i % 4]
    img = unnormalize(images_batch[i]).permute(1, 2, 0).cpu().numpy()
    label_idx = labels_batch[i].item()
    cls_code = CONFIG['class_names'][label_idx]
    cls_desc = CONFIG['class_fullnames'][label_idx]
    
    ax.imshow(img)
    ax.set_title(f"Label: {cls_code} ({label_idx})\n{cls_desc}", fontsize=10, fontweight='bold', color='darkblue')
    ax.axis('off')

plt.tight_layout()
plt.show()


## Langkah 5 — Inisialisasi Arsitektur Deep Learning Baseline (Transfer Learning)
Menyiapkan model baseline transfer learning standar dunia dermoskopi (misal: **ResNet-50** atau **EfficientNet-B0**) dengan bobot awal ImageNet dan modifikasi linear head menjadi 3 kelas output.


In [ ]:
def build_baseline_model(model_name="resnet50", num_classes=3, pretrained=True):
    """
    Membangun model deep learning baseline dengan pretrained weights.
    """
    if model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        print(f"🏗️ Inisialisasi Backbone: ResNet-50 (Head Linear: {in_features} -> {num_classes})")
        
    elif model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        print(f"🏗️ Inisialisasi Backbone: EfficientNet-B0 (Head Linear: {in_features} -> {num_classes})")
        
    else:
        raise ValueError(f"Model {model_name} belum didukung.")
        
    return model

# Inisialisasi Baseline Model ResNet-50
baseline_model = build_baseline_model(model_name="resnet50", num_classes=CONFIG['num_classes'], pretrained=True)
baseline_model = baseline_model.to(device)

# Loss Function dengan Penanganan Class Imbalance (Weighted Cross Entropy)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer AdamW & Learning Rate Scheduler
optimizer = optim.AdamW(baseline_model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=1e-6)

print(f"🎯 Loss Function: Weighted CrossEntropyLoss ({[f'{w:.2f}' for w in class_weights]})")
print(f"⚡ Optimizer: AdamW (lr={CONFIG['learning_rate']}, weight_decay={CONFIG['weight_decay']})")
print(f"📈 Scheduler: CosineAnnealingLR (T_max={CONFIG['epochs']})")


## Langkah 6 — Modular Training & Validation Engine dengan Model Checkpointing
Fungsi modular untuk melatih 1 epoch dan mengevaluasi validasi secara simultan.  
Metrik checkpointing utama adalah **Validation Macro F1-Score** (karena dataset memiliki ketimpangan kelas yang signifikan).


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Melatih model selama satu epoch penuh"""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_targets = []

    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    _, _, epoch_f1_macro, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)

    return epoch_loss, epoch_acc, epoch_f1_macro

def validate(model, dataloader, criterion, device):
    """Mengevaluasi performa model pada validation set"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for images, targets in dataloader:
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = running_loss / len(dataloader.dataset)
    val_acc = accuracy_score(all_targets, all_preds)
    val_bal_acc = balanced_accuracy_score(all_targets, all_preds)
    _, _, val_f1_macro, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)

    return val_loss, val_acc, val_bal_acc, val_f1_macro, np.array(all_targets), np.array(all_preds), np.array(all_probs)

print("🛠️ Fungsi Training & Validation Engine Siap.")


## Langkah 7 — Verifikasi Forward & Backward Pass (1 Batch Sanity Check)
Menguji coba 1 batch pada forward pass dan backward pass untuk memastikan gradien terhitung dengan tepat tanpa memory leak atau error dimensi tensor.


In [ ]:
# Sanity Check 1 Batch
sample_imgs, sample_lbls = next(iter(train_loader))
sample_imgs = sample_imgs.to(device)
sample_lbls = sample_lbls.to(device)

baseline_model.eval()
with torch.no_grad():
    sample_out = baseline_model(sample_imgs)
    sample_loss = criterion(sample_out, sample_lbls)

print("🧪 Sanity Check Forward Pass:")
print(f"   Input Tensor Shape  : {sample_imgs.shape}")
print(f"   Output Logits Shape : {sample_out.shape} (Batch x Classes)")
print(f"   Sample Loss Value   : {sample_loss.item():.4f}")
print("✅ Model dan pipeline siap dieksekusi untuk training penuh.")


## Langkah 8 — Evaluasi Komprehensif pada Test Set (Unseen Data)
Fungsi untuk mengevaluasi model pada Test Set independen (2.203 citra bebas kebocoran) dengan menghitung metrik lengkap:
* Overall Accuracy & Balanced Accuracy
* Per-Class Sensitivity (Recall), Specificity, Precision, & F1-Score
* Area Under the ROC Curve (Macro ROC-AUC One-vs-Rest)


In [ ]:
def evaluate_test_set(model, test_loader, criterion, device, class_names, class_fullnames):
    """
    Evaluasi klinis lengkap pada Test Set (Unseen Data).
    """
    test_loss, test_acc, test_bal_acc, test_f1_macro, targets, preds, probs = validate(
        model, test_loader, criterion, device
    )

    print("=" * 75)
    print("              LAPORAN EVALUASI PERFORMA MODEL PADA TEST SET")
    print("=" * 75)
    print(f"  Overall Accuracy        : {test_acc * 100:.2f}%")
    print(f"  Balanced Accuracy       : {test_bal_acc * 100:.2f}%")
    print(f"  Macro Average F1-Score  : {test_f1_macro * 100:.2f}%")
    print(f"  Test Loss               : {test_loss:.4f}")

    # One-vs-Rest ROC-AUC
    try:
        macro_roc_auc = roc_auc_score(targets, probs, multi_class='ovr', average='macro')
        print(f"  Macro ROC-AUC (OvR)     : {macro_roc_auc * 100:.2f}%")
    except Exception as e:
        macro_roc_auc = None
        print(f"  Macro ROC-AUC (OvR)     : N/A ({e})")

    print("-" * 75)
    print("  Rincian Metrik Diagnosis Klinis Per Kelas:")
    print("-" * 75)

    cm = confusion_matrix(targets, preds)
    per_class_p, per_class_r, per_class_f1, support = precision_recall_fscore_support(targets, preds, zero_division=0)

    rows = []
    for i, (code, desc) in enumerate(zip(class_names, class_fullnames)):
        # Specificity = TN / (TN + FP)
        tn = np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1))
        fp = np.sum(np.delete(cm[:, i], i))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        rows.append({
            "Kode": code,
            "Diagnosis": desc,
            "Jumlah Sampel": support[i],
            "Sensitivity (Recall)": f"{per_class_r[i]*100:.2f}%",
            "Specificity": f"{specificity*100:.2f}%",
            "Precision": f"{per_class_p[i]*100:.2f}%",
            "F1-Score": f"{per_class_f1[i]*100:.2f}%"
        })

    df_metrics = pd.DataFrame(rows)
    print(df_metrics.to_string(index=False))
    print("=" * 75)

    return {
        "loss": test_loss,
        "accuracy": test_acc,
        "balanced_accuracy": test_bal_acc,
        "f1_macro": test_f1_macro,
        "roc_auc_macro": macro_roc_auc,
        "targets": targets,
        "preds": preds,
        "probs": probs,
        "cm": cm,
        "metrics_df": df_metrics
    }

print("📋 Fungsi Evaluasi Komprehensif Medis Siap.")


## Langkah 9 — Visualisasi Diagnostik Klinis (Confusion Matrix & ROC Curve OvR)
Membangun grafik diagnostik klinis:
1. **Normalized Confusion Matrix Heatmap:** Menunjukkan akurasi prediksi vs ground truth riil untuk setiap kelas.
2. **Multi-Class ROC Curves (One-vs-Rest):** Menggambarkan Trade-off True Positive Rate vs False Positive Rate untuk masing-masing kelas target (`NV`, `MEL`, `BKL`).


In [ ]:
def plot_clinical_diagnostics(eval_results, class_names, class_fullnames):
    """
    Merender visualisasi Confusion Matrix dan Kurva ROC One-vs-Rest.
    """
    targets = eval_results['targets']
    preds = eval_results['preds']
    probs = eval_results['probs']
    cm = eval_results['cm']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 1. Normalized Confusion Matrix Heatmap
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    labels_display = [f"{c}\n({f})" for c, f in zip(class_names, class_fullnames)]

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        cbar=True,
        xticklabels=labels_display,
        yticklabels=labels_display,
        ax=axes[0],
        annot_kws={"size": 11, "weight": "bold"}
    )
    axes[0].set_title("Normalized Confusion Matrix (Test Set)", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Prediksi Model", fontsize=11, fontweight='bold')
    axes[0].set_ylabel("Ground Truth Medis", fontsize=11, fontweight='bold')

    # 2. Multi-Class ROC Curves (One-vs-Rest)
    colors = ['#2ca02c', '#d62728', '#1f77b4'] # NV=Green, MEL=Red (Kanker), BKL=Blue
    for i, (code, color) in enumerate(zip(class_names, colors)):
        y_true_binary = (targets == i).astype(int)
        fpr, tpr, _ = roc_curve(y_true_binary, probs[:, i])
        auc_score = roc_auc_score(y_true_binary, probs[:, i])
        axes[1].plot(fpr, tpr, color=color, lw=2.5, label=f"{code} (AUC = {auc_score*100:.1f}%)")

    axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Chance')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=11, fontweight='bold')
    axes[1].set_title('ROC Curves Multi-Class (One-vs-Rest)', fontsize=13, fontweight='bold')
    axes[1].legend(loc="lower right", fontsize=10, frameon=True)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

print("📈 Fungsi Visualisasi Diagnostik Medis Siap.")


## Langkah 10 — Prosedur Eksekusi Pelatihan Penuh (Training & Fine-Tuning)
Untuk memulai eksperimen pelatihan penuh:
1. Pastikan ketersediaan akselerasi GPU (lokal NVIDIA CUDA atau lingkungan Cloud seperti Google Colab / Kaggle).
2. Jalankan cell di bawah ini untuk memulai loop pelatihan hingga `CONFIG['epochs']`.
3. Checkpoint model terbaik (*Best Macro F1*) akan otomatis tersimpan di folder `models/best_baseline_resnet50_3kelas.pth`.


In [ ]:
def run_training_experiment(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device, save_path):
    """
    Menjalankan siklus training dan validasi lengkap dengan model checkpointing.
    """
    best_val_f1 = 0.0
    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_bal_acc': [], 'val_f1': []
    }

    print(f"🚀 Memulai Pelatihan Model Baseline ({num_epochs} Epochs) pada {device}...")
    start_total_time = time.time()

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_bal_acc, val_f1, _, _, _ = validate(model, val_loader, criterion, device)
        
        if scheduler:
            scheduler.step()

        elapsed = time.time() - epoch_start
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_bal_acc'].append(val_bal_acc)
        history['val_f1'].append(val_f1)

        print(f"Epoch [{epoch:02d}/{num_epochs:02d}] ({elapsed:.1f}s) | "
              f"Train Loss: {train_loss:.4f} - F1: {train_f1*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} - Acc: {val_acc*100:.2f}% - BalAcc: {val_bal_acc*100:.2f}% - F1: {val_f1*100:.2f}%")

        # Simpan checkpoint jika Validation Macro F1 meningkat
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_f1': val_f1,
                'config': CONFIG
            }, save_path)
            print(f"   ⭐ Checkpoint Disimpan! (Best Val Macro F1: {best_val_f1*100:.2f}%)")

    total_time = time.time() - start_total_time
    print(f"\n✅ Pelatihan Selesai dalam {total_time/60:.2f} menit. Model terbaik tersimpan di: {save_path}")
    return history

# Parameter Path Penyimpanan
    "checkpoint_dir": "../models" if os.path.exists("../Dataset") else "models",
print(f"💾 Path Checkpoint Terbaik: {BEST_MODEL_PATH}")
print("🏁 Pipeline eksperimen baseline 3 kelas siap dieksekusi.")
